# Parsing statistics

All numbers and figures on this page are computed when the book is built, from the two files that
`src/15_run_parsing.ipynb` writes to `logs/` at the end of every run: `m2_stats.json` (the results of
the run) and `m2_benchmark.json` (time, RAM and output size, added up over all Colab sessions). If a file
is not there yet, its sections say so and the rest of the page is still built. Definitions:

- **version status**: `ok` (tree and table rows written); `empty_source` (no `.tex` file: a PDF-only
  version or no `vN/` folder; recorded without rows, as the statement asks); `no_root` (`.tex` files but
  none with `\documentclass`); `error` (an exception, logged with its message);
- **parse success rate**: `ok` versions over the versions that have a `.tex` file, and over all the
  versions listed in `metadata.json`;
- **version coverage**: for each paper, the versions parsed over its versions on arXiv;
- **elements before / after deduplication**: every node of every version's tree, then the rows of the
  `elements` table (one per distinct element of a paper over all its versions). A leaf (`sentence`,
  `formula`, `figure`) is matched on `(kind, content)`, an inner node on `(kind, content, parent, k)`;
- **BibTeX entries before / after deduplication**: the entries extracted from each version, summed, then
  the rows of the `bibtex` table (one per reference of a paper).

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import report_data as data

stats, bench = data.stats(), data.benchmark()
display(Markdown(data.source_note(stats, bench)))
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML

PLOTLY_CONFIG = {"displaylogo": False, "responsive": True, "toImageButtonOptions": {"format": "png", "scale": 2}}


def show(fig, height=360):
    """An interactive figure in the built book: hover for the exact numbers, drag to zoom, double-click to reset,
    click a legend entry to hide it. plotly.js is loaded from its CDN."""
    fig.update_layout(template="plotly_white", height=height, margin=dict(l=10, r=10, t=60, b=10),
                      font=dict(size=12), hoverlabel=dict(font_size=12), title_x=0.0)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False, config=PLOTLY_CONFIG)))


BEFORE, AFTER = "#2a78d6", "#eb6834"   # slots 1 and 2 of a colour-blind-safe categorical palette
BAR = BEFORE                            # charts with a single series


def count(value):
    return "–" if value is None else f"{value:,}"


def pct(value):
    return "–" if value is None else f"{value:.2f}%"


def drop(before, after):
    return f"−{100 * (before - after) / before:.1f}%" if before else "–"


def table(rows, columns):
    display(pd.DataFrame(rows, columns=columns).style.hide(axis="index"))

## Summary

In [ ]:
rows = []
if stats:
    status, cover = stats.get("status", {}), stats.get("version_coverage", {})
    rows += [
        ("Papers parsed / papers in the run", f"{count(stats.get('papers_parsed'))} / {count(stats.get('papers'))}"),
        ("Versions in metadata.json / with a .tex file",
         f"{count(stats.get('versions'))} / {count(stats.get('versions_with_tex'))}"),
        ("Versions ok / empty_source / no_root / error", " / ".join(count(status.get(k, 0)) for k in data.STATUSES)),
        ("Parse success rate, versions with a .tex file", pct(stats.get("parse_success_rate_of_tex_versions_pct"))),
        ("Parse success rate, all versions", pct(stats.get("parse_success_rate_of_all_versions_pct"))),
        ("Papers with every / some / no version parsed",
         " / ".join(count(cover.get(k)) for k in ("papers_full", "papers_partial", "papers_none"))),
        ("Mean version coverage per paper", pct(cover.get("mean_pct"))),
        ("Elements before → after deduplication",
         f"{count(stats.get('elements_before_dedup'))} → {count(stats.get('elements_after_dedup'))} "
         f"({drop(stats.get('elements_before_dedup') or 0, stats.get('elements_after_dedup') or 0)})"),
        ("BibTeX entries before → after deduplication",
         f"{count(stats.get('bib_entries_before_dedup'))} → {count(stats.get('bib_entries_after_dedup'))} "
         f"({drop(stats.get('bib_entries_before_dedup') or 0, stats.get('bib_entries_after_dedup') or 0)})"),
    ]
if bench:
    wall = bench.get("wall_seconds") or 0
    per_paper, median = bench.get("avg_seconds_per_paper"), bench.get("median_paper_seconds_in_worker")
    peak, avg = bench.get("peak_ram_mb"), bench.get("avg_ram_mb")
    rows += [
        ("Sessions / total wall time", f"{bench.get('sessions', '–')} / {wall / 3600:,.2f} h"),
        ("Papers / versions in the benchmark", f"{count(bench.get('papers'))} / {count(bench.get('versions'))}"),
        ("Average time per paper (wall time / papers)", "–" if per_paper is None else f"{per_paper:.3f} s"),
        ("Median time per paper inside a worker", "–" if median is None else f"{median:.3f} s"),
        ("Peak RAM / time-weighted average RAM",
         f"{'–' if peak is None else f'{peak:,.0f} MB'} / {'–' if avg is None else f'{avg:,.0f} MB'}"),
        ("Output size (data/parsed + Parquet tables)",
         "–" if bench.get("output_bytes") is None else f"{bench['output_bytes'] / data.MB:,.1f} MB"),
        ("Parallel workers / CPU", f"{bench.get('workers', '–')} / {bench.get('colab_cpu', '–')}"),
    ]
if rows:
    table(rows, ["Statistic", "Value"])
else:
    display(Markdown(data.missing_note("stats") + "\n\n" + data.missing_note("benchmark")))

## Which notebook does each step

`13_dedup_export` uses the function of a member's notebook when it exists and its own fallback otherwise
(see its first cell). The run records which one each step used; the parser version says the same in short
(`10:mixed` means some functions of `10` come from the fallback).

In [ ]:
implementation = (stats or {}).get("implementation") or {}
if implementation:
    table(list(implementation.items()), ["Step", "Provided by"])
    display(Markdown(f"Parser version: `{stats.get('parser_version', '?')}`."))
else:
    display(Markdown(data.missing_note("stats")))

## Status of every version

In [ ]:
if stats:
    status = pd.Series({k: stats.get("status", {}).get(k, 0) for k in data.STATUSES})
    sources = pd.Series(stats.get("bib_source_of_tex_versions") or {}, dtype="int64").sort_values(ascending=False)
    charts = [(status, "Status of every version")] + ([(sources, "Source of the references")] if len(sources) else [])
    fig = make_subplots(rows=1, cols=len(charts), subplot_titles=[title for _, title in charts], horizontal_spacing=0.18)
    for col, (series, title) in enumerate(charts, start=1):
        total = max(int(series.sum()), 1)
        fig.add_trace(go.Bar(
            x=series.values, y=list(series.index), orientation="h", marker_color=BAR, name=title, showlegend=False,
            text=[f"{v:,} ({v / total:.1%})" for v in series.values], textposition="outside", cliponaxis=False,
            customdata=[v / total for v in series.values],
            hovertemplate="%{y}: %{x:,} versions (%{customdata:.2%})<extra></extra>"), row=1, col=col)
        fig.update_xaxes(title_text="Versions", range=[0, max(int(series.max()), 1) * 1.45], row=1, col=col)
        fig.update_yaxes(autorange="reversed", row=1, col=col)
    show(fig, height=320)
    reasons = stats.get("empty_source_reasons") or {}
    with_tex = stats.get("versions_with_tex") or 0
    display(Markdown(
        f"Of **{count(stats.get('versions'))}** versions, **{count(with_tex)}** have a `.tex` file and "
        f"**{count(status['ok'])}** of those were parsed ({pct(stats.get('parse_success_rate_of_tex_versions_pct'))}). "
        f"The other versions are `empty_source`: "
        + (", ".join(f"{reason or 'no reason logged'}: **{n:,}**" for reason, n in reasons.items()) or "none")
        + ". They are part of the corpus (the statement keeps PDF-only versions as empty records) but have no "
          "LaTeX to parse. The source of the references is counted over the versions with a `.tex` file: "
          "`bib` (a `.bib` file), `bibitem` (`thebibliography` in the `.tex`), `missing_bbl` (a bibliography is "
          "declared but its `.bib` was not shipped: the references were only in the `.bbl`, which the "
          "download of Milestone 1 dropped) and `none`."))
else:
    display(Markdown(data.missing_note("stats")))

## Version coverage per paper

In [ ]:
distribution = ((stats or {}).get("version_coverage") or {}).get("distribution") or {}
if distribution:
    keys = list(distribution)   # sorted by the number of versions, then parsed / versions
    fig = go.Figure(go.Bar(
        x=keys, y=[distribution[k] for k in keys], marker_color=BAR, text=[f"{distribution[k]:,}" for k in keys],
        textposition="outside", cliponaxis=False,
        hovertemplate="%{x} versions parsed: %{y:,} papers<extra></extra>"))
    fig.update_layout(title="Version coverage per paper", xaxis_title="Versions parsed / versions on arXiv",
                      yaxis_title="Papers (log scale)", yaxis_type="log", xaxis_type="category")
    show(fig)
    cover = stats["version_coverage"]
    display(Markdown(
        f"**{count(cover.get('papers_full'))}** papers have every version parsed, **{count(cover.get('papers_partial'))}** "
        f"some of them and **{count(cover.get('papers_none'))}** none (no version with LaTeX, or none parsed); the mean "
        f"coverage is **{pct(cover.get('mean_pct'))}** of the versions of a paper."))
else:
    display(Markdown(data.missing_note("stats")))

## Elements and BibTeX entries before and after deduplication

In [ ]:
before = pd.Series((stats or {}).get("elements_before_dedup_by_kind") or {}, dtype="int64")
after = pd.Series((stats or {}).get("elements_after_dedup_by_kind") or {}, dtype="int64")
if len(before) or len(after):
    kinds = before.add(after, fill_value=0).sort_values(ascending=False).index
    frame = pd.DataFrame({"before": before.reindex(kinds).fillna(0).astype(int),
                          "after": after.reindex(kinds).fillna(0).astype(int)})
    frame["removed"] = [drop(b, a) for b, a in zip(frame["before"], frame["after"])]
    fig = go.Figure()
    for column, colour, name in (("before", BEFORE, "before: every node of every version"),
                                 ("after", AFTER, "after: rows of elements")):
        fig.add_trace(go.Bar(
            y=list(kinds), x=frame[column].clip(lower=1), orientation="h", name=name, marker_color=colour,
            customdata=frame[["before", "after", "removed"]].values,
            hovertemplate="%{y}<br>before %{customdata[0]:,} · after %{customdata[1]:,} (%{customdata[2]})"
                          "<extra></extra>"))
    fig.update_layout(title="Elements per kind, before and after deduplication", barmode="group",
                      xaxis_title="Count (log scale)", xaxis_type="log", yaxis_autorange="reversed",
                      legend=dict(orientation="h", y=-0.15))
    show(fig, height=40 * len(kinds) + 180)
    total = ("all kinds", int(frame["before"].sum()), int(frame["after"].sum()),
             drop(frame["before"].sum(), frame["after"].sum()))
    rows = [(kind, f"{b:,}", f"{a:,}", r) for kind, b, a, r in frame.itertuples()]
    table(rows + [(total[0], f"{total[1]:,}", f"{total[2]:,}", total[3])],
          ["Kind", "Before (nodes)", "After (elements)", "Removed"])
    bib_before, bib_after = stats.get("bib_entries_before_dedup") or 0, stats.get("bib_entries_after_dedup") or 0
    display(Markdown(
        f"Deduplication turns **{total[1]:,}** nodes into **{total[2]:,}** elements "
        f"(**{drop(total[1], total[2]).lstrip('−')}** fewer): a sentence, formula or figure that "
        f"is identical in two versions of a paper (after cleaning, and after the citation keys were unified) "
        f"is one element with one `hierarchy` row per version. The `document` row is one per paper, shared by "
        f"its versions. BibTeX: **{bib_before:,}** entries extracted over all versions become **{bib_after:,}** "
        f"rows of `bibtex` ({drop(bib_before, bib_after)}): two entries are one reference when they share the "
        f"key, the DOI, the arXiv ID or the normalised title, inside a version or across versions."))
else:
    display(Markdown(data.missing_note("stats")))

## Running time, RAM and output size

In [ ]:
series = (bench or {}).get("ram_timeseries") or []
if bench:
    if series:
        seconds, mb = zip(*series)
        fig = go.Figure(go.Scatter(x=np.array(seconds) / 60, y=mb, mode="lines", line=dict(color=BAR, width=2),
                                   hovertemplate="minute %{x:,.1f}: %{y:,.0f} MB<extra></extra>"))
        fig.update_layout(title="RAM during the run",
                          xaxis_title="Minutes since the first sample (sessions one after another)",
                          yaxis_title="RSS of the parser and its workers (MB)", yaxis_rangemode="tozero",
                          xaxis_rangeslider_visible=True)
        show(fig, height=420)
    else:
        display(Markdown("*No RAM samples in `m2_benchmark.json`.*"))
    wall, papers = bench.get("wall_seconds") or 0, bench.get("papers") or 0
    output = bench.get("output_bytes")
    display(Markdown(
        f"**{count(papers)}** papers ({count(bench.get('versions'))} versions) took **{wall / 3600:,.2f} h** of wall "
        f"time over **{bench.get('sessions', '–')}** session(s) with **{bench.get('workers', '–')}** worker "
        f"process(es) on `{bench.get('colab_cpu', '?')}`: **{'–' if not papers else f'{wall / papers:.3f}'} s** "
        f"per paper on average. Peak RAM was **{bench.get('peak_ram_mb') or '–'} MB** (the parser and its "
        f"workers, sampled every 2 s) and the output, `data/parsed/` plus the three Parquet tables, takes "
        f"**{'–' if output is None else f'{output / data.MB:,.1f} MB'}**."))
else:
    display(Markdown(data.missing_note("benchmark")))